# Aerial Object Adversarial Evaluation Workflow: Solution

This solution evaluates a CIFAR-10 airplane binary classifier under a PGD evasion attack across three epsilon values. The `airplane` class stands in for an unauthorized aerial object in the security monitoring scenario.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

EXERCISE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path("module-7-apply-evasion-attacks/exercise/solution")
ART_HOME = EXERCISE_ROOT / "art_data"
ART_HOME.mkdir(parents=True, exist_ok=True)
os.environ["ART_DATA_PATH"] = str(ART_HOME)
os.environ["HOME"] = str(EXERCISE_ROOT)
os.environ["USERPROFILE"] = str(EXERCISE_ROOT)
sys.path.append(str(EXERCISE_ROOT / "src"))

from aerial_eval_utils import (
    CLASS_NAMES,
    attack_success_rate,
    build_art_classifier,
    evaluate_numpy,
    load_aerial_dataset,
    make_attack,
    perturbation_metrics,
    plot_comparison_grid,
    predict,
    save_metrics_table,
    train_or_load_model,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Load Model and Data

In [ ]:
data_dir = EXERCISE_ROOT / "data/generated"
model_path = EXERCISE_ROOT / "models/aerial_object_cnn.pt"

model = train_or_load_model(model_path, data_dir, device=device, force_train=False)
val_dataset = load_aerial_dataset(data_dir, split="val")
val_images = val_dataset.images
val_labels = val_dataset.labels

clean_metrics = evaluate_numpy(model, val_images, val_labels, device=device)
print(f"Clean accuracy: {clean_metrics['accuracy']:.3f}")
print(f"Mean clean confidence: {clean_metrics['mean_confidence']:.3f}")
print(f"Clean false negative rate: {clean_metrics['false_negative_rate']:.3f}")

## Configure Attack

In [ ]:
attack_name = "pgd"
epsilon_values = [0.01, 0.03, 0.06]
classifier = build_art_classifier(model)

evaluation_images = val_images[:120]
evaluation_labels = val_labels[:120]
clean_eval = evaluate_numpy(model, evaluation_images, evaluation_labels, device=device)
clean_predictions = clean_eval["predictions"]

rows = []
adversarial_examples_by_epsilon = {}

for epsilon in epsilon_values:
    attack = make_attack(attack_name, classifier, epsilon)
    adversarial_images = attack.generate(x=evaluation_images)
    adversarial_examples_by_epsilon[epsilon] = adversarial_images

    adv_eval = evaluate_numpy(model, adversarial_images, evaluation_labels, device=device)
    adv_predictions = adv_eval["predictions"]
    perturb = perturbation_metrics(evaluation_images, adversarial_images)

    row = {
        "attack": attack_name,
        "epsilon": epsilon,
        "clean_accuracy": round(clean_eval["accuracy"], 4),
        "adversarial_accuracy": round(adv_eval["accuracy"], 4),
        "attack_success_rate": round(attack_success_rate(clean_predictions, adv_predictions, evaluation_labels), 4),
        "mean_clean_confidence": round(clean_eval["mean_confidence"], 4),
        "mean_adversarial_confidence": round(adv_eval["mean_confidence"], 4),
        "mean_linf": round(perturb["linf"], 4),
        "mean_l2": round(perturb["l2_mean"], 4),
    }
    rows.append(row)

metrics_path = save_metrics_table(rows, EXERCISE_ROOT / "results/adversarial_metrics.csv")
print(f"Saved metrics to: {metrics_path}")
rows

## Visualize Highest-Epsilon Result

In [ ]:
selected_epsilon = epsilon_values[-1]
selected_adv = adversarial_examples_by_epsilon[selected_epsilon]
adv_predictions, adv_confidence, _ = predict(model, selected_adv, device=device)

comparison_path = plot_comparison_grid(
    evaluation_images,
    selected_adv,
    clean_eval["predictions"],
    adv_predictions,
    evaluation_labels,
    EXERCISE_ROOT / f"results/{attack_name}_epsilon_{selected_epsilon}_comparison.png",
)
print(f"Saved comparison grid to: {comparison_path}")

## Robustness Assessment

Review `results/adversarial_metrics.csv` and `docs/robustness_assessment.md` for the final report structure.